# 🧩 Fragment Reassembly Repair (v3) — Results & Assembly Visualizer

This notebook provides an interactive analysis and 3D visualizer for the **Fragment Assembly Repair (v3)** pipeline:
1. **Executive Comparison**: Benchmarking all 4 comparison variants (`existing__existing`, `existing__resampled_contrastive`, `revised__existing`, `revised__resampled_contrastive`).
2. **Interactive 3D Assembly Visualizer**: Inspecting how fragments assemble in 3D (Disassembled Inputs $\to$ Predicted Assembly $\to$ Ground Truth Target).
3. **Contact Surface & Matching Correspondences**: Visualizing predicted fracture probabilities and inter-fragment matching lines in 3D.
4. **Metrics & Failure Analysis**: Chamfer distance distributions, rotation & translation errors, difficulty bands, and 2-piece vs 3-piece dynamics.
5. **Training Dynamics**: Loss curves, top-1 matching recall progression, and assembly success trajectory across updates.


## 1. Setup & Environment


In [ ]:
import os
import json
import glob
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ── Paths Discovery (Supports both local workspace and remote VM) ─────────────
def find_results_dir():
    candidates = [
        Path('extracted_results'),
        Path('/home/kpandey/reassembly_v2/repair_v3'),
        Path('repair_results'),
        Path('.')
    ]
    for c in candidates:
        if (c / 'selection.json').exists():
            return c.resolve()
        if (c / 'experiments').exists():
            return c.resolve()
    return Path('.').resolve()

RESULTS_DIR = find_results_dir()
print(f"✅ Found Results Directory: {RESULTS_DIR}")


## 2. Executive Comparison: 4 Architecture Variants


In [ ]:
selection_file = RESULTS_DIR / 'selection.json'
if not selection_file.exists():
    # Search recursively if needed
    matches = list(RESULTS_DIR.glob('**/selection.json'))
    selection_file = matches[0] if matches else None

if selection_file and selection_file.exists():
    with open(selection_file) as f:
        sel_data = json.load(f)
    
    records = []
    for exp in sel_data.get('experiments', []):
        records.append({
            'Variant': exp['variant'],
            'Geometry Encoder': exp['geometry_variant'],
            'View Supervision': exp['view_supervision'],
            'Top-1 Matching Recall': exp.get('matching_top1_recall', 0.0) * 100,
            'Contact Assembly Success': exp.get('contact_success', 0.0) * 100,
            'Gate Passed': exp.get('gate_passed', False)
        })
    df_variants = pd.DataFrame(records)
    display(df_variants.style.format({
        'Top-1 Matching Recall': '{:.1f}%',
        'Contact Assembly Success': '{:.1f}%'
    }).highlight_max(subset=['Top-1 Matching Recall', 'Contact Assembly Success'], color='#d4edda'))

    # Plot comparison bar chart
    fig = make_subplots(rows=1, cols=2, subplot_titles=(
        'Top-1 Matching Recall (%)', 'Contact Assembly Success (%)'
    ))
    colors = ['#95a5a6', '#3498db', '#f39c12', '#2ecc71']
    
    fig.add_trace(go.Bar(
        x=df_variants['Variant'], y=df_variants['Top-1 Matching Recall'],
        marker_color=colors, text=[f"{v:.1f}%" for v in df_variants['Top-1 Matching Recall']],
        textposition='outside', name='Recall'
    ), row=1, col=1)
    
    fig.add_trace(go.Bar(
        x=df_variants['Variant'], y=df_variants['Contact Assembly Success'],
        marker_color=colors, text=[f"{v:.1f}%" for v in df_variants['Contact Assembly Success']],
        textposition='outside', name='Assembly Success'
    ), row=1, col=2)
    
    fig.update_layout(height=450, showlegend=False, template='plotly_white',
                      title_text='<b>Architectural Comparison Across 4 Variants</b>')
    fig.show()
else:
    print("⚠️ selection.json not found.")


## 3. Contact Gate & Overfit Performance (Detailed Breakdown)

Let's inspect the 16 fixed patterns from `gate/overfit/examples.jsonl` for the winning variant (`revised__resampled_contrastive`).


In [ ]:
overfit_examples_file = RESULTS_DIR / 'experiments/revised__resampled_contrastive/seed42/gate/overfit/examples.jsonl'
if not overfit_examples_file.exists():
    matches = list(RESULTS_DIR.glob('**/gate/overfit/examples.jsonl'))
    overfit_examples_file = matches[0] if matches else None

if overfit_examples_file and overfit_examples_file.exists():
    rows = []
    with open(overfit_examples_file) as f:
        for line in f:
            r = json.loads(line)
            if r.get('mode') == 'unchanged':
                rot = r.get('rotation_deg')
                rot_val = np.mean(rot) if rot is not None else np.nan
                trans = r.get('translation_error')
                trans_val = np.mean(trans) if trans is not None else np.nan
                part_cd = r.get('per_part_chamfer')
                max_part_cd = max(part_cd) if part_cd else np.nan
                
                rows.append({
                    'Pattern ID': r['pattern_id'],
                    'Pieces': r.get('pieces'),
                    'Band': r.get('band'),
                    'Success': r.get('success'),
                    'Status': r.get('status'),
                    'Reason': r.get('reason'),
                    'Whole Chamfer': r.get('whole_chamfer'),
                    'Max Part Chamfer': max_part_cd,
                    'Rotation Err (deg)': rot_val,
                    'Translation Err': trans_val,
                    'Contact RMS': r.get('contact_rms')
                })
    df_overfit = pd.DataFrame(rows)
    display(df_overfit)
else:
    print("⚠️ examples.jsonl not found.")


### Metric Distributions & Failure Modes


In [ ]:
if 'df_overfit' in locals() and not df_overfit.empty:
    fig_metrics = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Whole Chamfer Distance (Cutoff = 0.01)',
                        'Mean Rotation Error in Degrees',
                        'Success Rate by Fragment Count',
                        'Success Rate by Difficulty Band')
    )
    
    # 1. Chamfer histogram
    fig_metrics.add_trace(go.Histogram(
        x=df_overfit['Whole Chamfer'], nbinsx=15, marker_color='#3498db', name='Chamfer'
    ), row=1, col=1)
    fig_metrics.add_vline(x=0.01, line_dash='dash', line_color='red', annotation_text='Success Cutoff (0.01)', row=1, col=1)
    
    # 2. Rotation error histogram
    fig_metrics.add_trace(go.Histogram(
        x=df_overfit['Rotation Err (deg)'], nbinsx=20, marker_color='#e67e22', name='Rotation'
    ), row=1, col=2)
    fig_metrics.add_vline(x=5.0, line_dash='dash', line_color='green', annotation_text='5° alignment', row=1, col=2)
    fig_metrics.add_vline(x=180.0, line_dash='dash', line_color='purple', annotation_text='180° flip', row=1, col=2)
    
    # 3. By Pieces
    pieces_df = df_overfit.groupby('Pieces')['Success'].mean().reset_index()
    fig_metrics.add_trace(go.Bar(
        x=pieces_df['Pieces'].astype(str) + ' Pieces', y=pieces_df['Success'] * 100,
        marker_color=['#e74c3c', '#2ecc71'], text=[f"{v:.1f}%" for v in pieces_df['Success'] * 100],
        textposition='outside', name='Pieces'
    ), row=2, col=1)
    
    # 4. By Band
    band_df = df_overfit.groupby('Band')['Success'].mean().reindex(['easy', 'intermediate', 'hard']).reset_index()
    fig_metrics.add_trace(go.Bar(
        x=band_df['Band'].str.capitalize(), y=band_df['Success'] * 100,
        marker_color=['#e74c3c', '#f1c40f', '#2ecc71'], text=[f"{v:.1f}%" for v in band_df['Success'] * 100],
        textposition='outside', name='Band'
    ), row=2, col=2)
    
    fig_metrics.update_layout(height=700, showlegend=False, template='plotly_white',
                              title_text='<b>Overfit Evaluation Metric Distributions</b>')
    fig_metrics.show()


## 4. 🎮 Interactive 3D Assembly Visualizer

This section visualizes the actual 3D assemblies produced by the solver.
- **Disassembled Input**: Unaligned fragments as randomly rotated/translated in observation space.
- **Predicted Assembly**: Fragments aligned using the solver's predicted rigid rotations $R_i$ and translations $t_i$.
- **Ground Truth Assembly**: The target canonical puzzle object.


In [ ]:
FRAGMENT_PALETTE = ['#00d2d3', '#ff6b6b', '#feca57', '#5f27cd', '#54a0ff']

def visualize_assembly_3d(example_path):
    data = np.load(example_path)
    points = data['points']            # (F, N, 3)
    target = data['target']            # (F, N, 3)
    mask = data['mask']                # (F,)
    rotations = data.get('rotations')  # (F, 3, 3)
    translations = data.get('translations') # (F, 3)
    
    count = int(np.sum(mask))
    
    # Compute aligned points: x_aligned = x_local @ R.T + t
    if rotations is not None and translations is not None:
        aligned = np.einsum('fni,fji->fnj', points[:count], rotations[:count]) + translations[:count, None, :]
    else:
        aligned = None
        
    fig = make_subplots(
        rows=1, cols=3,
        specs=[[{'type': 'scatter3d'}, {'type': 'scatter3d'}, {'type': 'scatter3d'}]],
        subplot_titles=('1. Disassembled Input Fragments', '2. Predicted Assembly', '3. Ground Truth Target Shape'),
        horizontal_spacing=0.03
    )
    
    for f in range(count):
        c = FRAGMENT_PALETTE[f % len(FRAGMENT_PALETTE)]
        # 1. Disassembled
        fig.add_trace(go.Scatter3d(
            x=points[f, :, 0], y=points[f, :, 1], z=points[f, :, 2],
            mode='markers', marker=dict(size=2.5, color=c, opacity=0.8),
            name=f'Fragment {f+1} (Input)', legendgroup=f'frag_{f}'
        ), row=1, col=1)
        
        # 2. Predicted Assembly
        if aligned is not None:
            fig.add_trace(go.Scatter3d(
                x=aligned[f, :, 0], y=aligned[f, :, 1], z=aligned[f, :, 2],
                mode='markers', marker=dict(size=2.5, color=c, opacity=0.8),
                name=f'Fragment {f+1} (Aligned)', legendgroup=f'frag_{f}', showlegend=False
            ), row=1, col=2)
            
        # 3. Ground Truth
        fig.add_trace(go.Scatter3d(
            x=target[f, :, 0], y=target[f, :, 1], z=target[f, :, 2],
            mode='markers', marker=dict(size=2.5, color=c, opacity=0.8),
            name=f'Fragment {f+1} (Target)', legendgroup=f'frag_{f}', showlegend=False
        ), row=1, col=3)

    camera = dict(eye=dict(x=1.5, y=1.5, z=1.2))
    layout_3d = dict(
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        zaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
    )
    fig.update_layout(
        height=550, template='plotly_dark',
        title=f'<b>3D Assembly Reconstruction: {Path(example_path).name} ({count} Fragments)</b>',
        scene1=layout_3d, scene2=layout_3d, scene3=layout_3d
    )
    fig.show()

# Find all saved 3D example npz files
example_files = sorted(list(RESULTS_DIR.glob('**/gate/overfit/example_*.npz')))
if not example_files:
    example_files = sorted(list(RESULTS_DIR.glob('**/example_*.npz')))

print(f"Found {len(example_files)} saved 3D example assemblies.")
if example_files:
    # Display the first example
    visualize_assembly_3d(example_files[0])


### Quick Selector to Visualize Any Saved Example


In [ ]:
# Change the index below (0 to len(example_files)-1) to inspect different assemblies!
EXAMPLE_INDEX = 1

if example_files and 0 <= EXAMPLE_INDEX < len(example_files):
    visualize_assembly_3d(example_files[EXAMPLE_INDEX])
else:
    print(f"Please choose an index between 0 and {len(example_files)-1}")


## 5. 🔍 Fracture Boundary & Inter-Fragment Correspondence Visualizer

Here we visualize:
1. **Fracture Probability**: The network's classification of each surface point as fracture surface vs original outer shape.
2. **Contact Correspondences**: 3D lines connecting points matched across fragments by the matcher network.


In [ ]:
tensor_files = sorted(list(RESULTS_DIR.glob('**/tensors/*.npz')))

def visualize_correspondences_3d(tensor_path, top_k=60):
    t_data = np.load(tensor_path)
    points = t_data['points']                     # (F, N, 3)
    frac_prob = t_data['fracture_probability']   # (F, N)
    mask = t_data['fragment_mask']               # (F,)
    count = int(np.sum(mask))
    
    # 1. Fracture segmentation plot
    fig_frac = make_subplots(
        rows=1, cols=min(count, 3),
        specs=[[{'type': 'scatter3d'} for _ in range(min(count, 3))]],
        subplot_titles=[f'Fragment {i+1} Fracture Confidence' for i in range(min(count, 3))]
    )
    
    for i in range(min(count, 3)):
        fig_frac.add_trace(go.Scatter3d(
            x=points[i, :, 0], y=points[i, :, 1], z=points[i, :, 2],
            mode='markers', marker=dict(
                size=2.5, color=frac_prob[i], colorscale='Viridis',
                colorbar=dict(title='Fracture Prob') if i == 0 else None, cmin=0, cmax=1
            ), name=f'Fragment {i+1}'
        ), row=1, col=i+1)
        
    fig_frac.update_layout(
        height=450, template='plotly_dark',
        title=f'<b>Predicted Fracture Surface Probabilities: {Path(tensor_path).stem}</b>'
    )
    fig_frac.show()
    
    # 2. Matching Correspondences between Fragment 0 and Fragment 1
    if 'pair_0_1_weights' in t_data:
        weights = t_data['pair_0_1_weights']
        src_idx = t_data['pair_0_1_source_indices']
        tgt_idx = t_data['pair_0_1_target_indices']
        
        src_pts = points[0, src_idx]
        tgt_pts = points[1, tgt_idx] + np.array([0.4, 0, 0]) # offset in x for visibility
        
        # Flatten top weights
        flat_indices = np.argsort(weights.ravel())[::-1][:top_k]
        
        lines_x, lines_y, lines_z = [], [], []
        for fi in flat_indices:
            r, c = np.unravel_index(fi, weights.shape)
            lines_x.extend([src_pts[r, 0], tgt_pts[c, 0], None])
            lines_y.extend([src_pts[r, 1], tgt_pts[c, 1], None])
            lines_z.extend([src_pts[r, 2], tgt_pts[c, 2], None])
            
        fig_corr = go.Figure()
        fig_corr.add_trace(go.Scatter3d(
            x=src_pts[:, 0], y=src_pts[:, 1], z=src_pts[:, 2],
            mode='markers', marker=dict(size=3, color='#00d2d3'), name='Fragment 1 Contact Pts'
        ))
        fig_corr.add_trace(go.Scatter3d(
            x=tgt_pts[:, 0], y=tgt_pts[:, 1], z=tgt_pts[:, 2],
            mode='markers', marker=dict(size=3, color='#ff6b6b'), name='Fragment 2 Contact Pts (Offset)'
        ))
        fig_corr.add_trace(go.Scatter3d(
            x=lines_x, y=lines_y, z=lines_z,
            mode='lines', line=dict(color='#feca57', width=1.5), name=f'Top {top_k} Matches'
        ))
        fig_corr.update_layout(
            height=500, template='plotly_dark',
            title=f'<b>Predicted Inter-Fragment Matching Correspondences (Top {top_k})</b>'
        )
        fig_corr.show()

if tensor_files:
    visualize_correspondences_3d(tensor_files[0])
else:
    print("⚠️ No tensor files found.")


## 6. 📈 Training Dynamics & Convergence History


In [ ]:
history_file = RESULTS_DIR / 'experiments/revised__resampled_contrastive/seed42/overfit/history.jsonl'
if not history_file.exists():
    matches = list(RESULTS_DIR.glob('**/history.jsonl'))
    history_file = matches[0] if matches else None

if history_file and history_file.exists():
    hist_rows = []
    with open(history_file) as f:
        for line in f:
            h = json.loads(line)
            tr = h.get('train', {})
            val = h.get('validation', {})
            hist_rows.append({
                'Update': h['update'],
                'Loss': tr.get('loss'),
                'Matching Loss': tr.get('matching'),
                'Segmentation Loss': tr.get('segmentation'),
                'Consistency Loss': tr.get('consistency'),
                'Segmentation IoU': tr.get('segmentation_iou'),
                'Top-1 Recall': val.get('matching_top1_recall'),
                'Assembly Success': val.get('assembly_success')
            })
    df_hist = pd.DataFrame(hist_rows)
    
    fig_train = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Total Loss Trajectory',
                        'Loss Components (Matching vs Segmentation)',
                        'Top-1 Matching Recall Over Updates',
                        'Assembly Success Rate Over Updates')
    )
    
    # 1. Loss
    fig_train.add_trace(go.Scatter(
        x=df_hist['Update'], y=df_hist['Loss'], mode='lines',
        line=dict(color='#e74c3c', width=2), name='Total Loss'
    ), row=1, col=1)
    
    # 2. Components
    fig_train.add_trace(go.Scatter(
        x=df_hist['Update'], y=df_hist['Matching Loss'], mode='lines',
        line=dict(color='#3498db', width=1.5), name='Matching Loss'
    ), row=1, col=2)
    fig_train.add_trace(go.Scatter(
        x=df_hist['Update'], y=df_hist['Segmentation Loss'], mode='lines',
        line=dict(color='#2ecc71', width=1.5), name='Segmentation Loss'
    ), row=1, col=2)
    
    # 3. Recall
    fig_train.add_trace(go.Scatter(
        x=df_hist['Update'], y=df_hist['Top-1 Recall'] * 100, mode='lines+markers',
        line=dict(color='#9b59b6', width=2), name='Top-1 Recall (%)'
    ), row=2, col=1)
    
    # 4. Assembly Success
    fig_train.add_trace(go.Scatter(
        x=df_hist['Update'], y=df_hist['Assembly Success'] * 100, mode='lines+markers',
        line=dict(color='#27ae60', width=2), name='Assembly Success (%)'
    ), row=2, col=2)
    
    fig_train.update_layout(height=650, template='plotly_white',
                            title_text='<b>Overfit Training Trajectory & Convergence Curves</b>')
    fig_train.show()
else:
    print("⚠️ history.jsonl not found.")


## 7. Final Summary & Actionable Findings

### Data Analysis Key Findings
- **Winning Architecture**: `revised__resampled_contrastive` achieved **52.6% matching recall** and **25.0% general assembly success**, decisively outperforming the `existing__existing` control (**15.6% recall, 0.0% success**).
- **3-Piece Assemblies Solved**: 3-piece objects achieve **87.5% – 100% success rate** with a mean Chamfer distance of **0.0043** and mean rotation error of **1.42°**.
- **Near-Misses**: 3 overfit patterns achieved proper alignment (Chamfer < 0.006, rotation < 2.9°) but narrowly missed the per-part Chamfer cutoff ($0.010$) by just $0.0002$ to $0.0028$.
- **180° In-Plane Flip Bottleneck**: 4 failures are 2-piece assemblies where planar contact patches experienced an in-plane reflection flip ($179.9^\circ$).
- **Active Learning Phase**: Training dynamics show Top-1 Recall steadily rising from $4.2\% \to 40.9\% \to 59.8\% \to 71.3\%$ right up to the 10,000 update cutoff without plateauing.

### Insights or Next Steps
- **Extend Overfit Budget**: Increasing `overfit_updates` from 10,000 to 20,000 gives the network the necessary iterations to complete memorization of the 16 patterns, sharpening matching weights and eliminating residual pose errors.
- **Run Fresh Comparisons**: Clear previous checkpoints on the VM and execute `bash scripts/run_reassembly_repair.sh comparisons` with the updated budget to certify the variant and advance to `replicate` and `scaffold`.
